Group Member 1: metrics + bootstrap 95% AUROC CI + combined evaluation function

In [ ]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    
    y_pred = (y_prob >= threshold).astype(int)

    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "AUROC": auroc,
        "AUPRC": auprc,
        "F1": f1,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Accuracy": accuracy
    }

def bootstrap_auroc_ci(
    y_true,
    y_prob,
    n_bootstraps=1000,
    random_state=42
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    
    rng = np.random.default_rng(random_state)
    scores = []
    n_samples = len(y_true)

    for _ in range(n_bootstraps):
        indices = rng.choice(
            n_samples,
            size=n_samples,
            replace=True
        )

        y_sample = y_true[indices]
        prob_sample = y_prob[indices]

        if len(np.unique(y_sample)) < 2:
            continue

        score = roc_auc_score(y_sample, prob_sample)
        scores.append(score)

    if not scores:
        return 0.0, 0.0

    lower, upper = np.percentile(scores, [2.5, 97.5])
    return lower, upper

def evaluate_model(y_true, y_prob):
    metrics = calculate_metrics(y_true, y_prob)
    lower, upper = bootstrap_auroc_ci(y_true, y_prob)

    metrics["AUROC_CI_Lower"] = lower
    metrics["AUROC_CI_Upper"] = upper

    return metrics


Group Member 2: confusion matrix + ROC curve + precision-recall curve + plotting helper

In [ ]:
def make_results_table(results):

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(
        "AUROC",
        ascending=False
    )

    return results_df

In [ ]:
def plot_model_comparison(results_df):

    labels = (
        results_df["Feature Set"]
        + " - "
        + results_df["Model"]
    )

    plt.figure(figsize=(9, 5))

    plt.bar(
        labels,
        results_df["AUROC"]
    )

    plt.ylabel("AUROC")
    plt.title("Model Performance Comparison")

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.tight_layout()
    plt.show()